In [1]:
import re

import pandas as pd
import numpy as np
from pathlib import Path


from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

from datasets import Dataset, load_dataset, DatasetDict, Features, ClassLabel, Value
import evaluate
import torch
import os

In [2]:
def trainit(i, df_training, df_predicting, ls_subsegment, output_dir = Path('PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_BERT/data_patent')):
    SELECTED_SUBSEGMENT = ls_subsegment[i]

    # drop if patent_abstract is empty
    df_training = df_training[df_training['patent_abstract'].str.len() > 0]
    # Generate label
    df_training.loc[:, 'label'] = 0
    df_training.loc[df_training['fullname'] == SELECTED_SUBSEGMENT, 'label'] = 1

    PARENT_SEGMENT = df_training[df_training['fullname'] == SELECTED_SUBSEGMENT]['segment'].values[0]
    print('working on subsegment:', SELECTED_SUBSEGMENT)
    # ## Processing Negative Samples: Filter negative samples from adjacent subsegments
    ds_positives = df_training[df_training.label == 1]
    len_positives = len(ds_positives)

    # if len_positive < 50, skip the training
    if len_positives < 20:
        print('Skip the training for subsegment:', SELECTED_SUBSEGMENT)
        return
    
    # if len_positive > 1000, downsample
    if len_positives > 500:
        ds_positives = ds_positives.sample(500, replace=True)
        len_positives = 500

    # num_sampled is the integer number of 10% of the positive samples plus all patents from adjacent subsegments
    num_sampled = int(len_positives * 0.05)
    ds_negatives_adjacent_subsegments = df_training[(df_training.label == 0) & (df_training.segment == PARENT_SEGMENT)]
    ds_negatives_downsample = df_training[df_training.label == 0].groupby('segment', group_keys=False).apply(lambda x: x.sample(num_sampled, replace=True)) 
    # make sure ds_negatives_adjacent_subsegments is not larger than ds_negatives_downsample
    if len(ds_negatives_adjacent_subsegments) > len(ds_negatives_downsample):
        ds_negatives_adjacent_subsegments = ds_negatives_adjacent_subsegments.sample(len(ds_negatives_downsample), replace=True)
    ds_negatives = pd.concat([ds_negatives_adjacent_subsegments, ds_negatives_downsample]).drop_duplicates().reset_index(drop=True)


    ## Train test split
    negativeTrain, negativeValidTest = train_test_split(ds_negatives, test_size=0.2)
    negativeValid, negativeTest = train_test_split(negativeValidTest, test_size=0.5)
    positiveTrain, positiveValidTest = train_test_split(ds_positives, test_size = 0.2)
    positiveValid, positiveTest = train_test_split(positiveValidTest, test_size=0.5)
    TrainDf = pd.concat([negativeTrain, positiveTrain]).drop(['companyid','marketmap', 'segment', 'subsegment'], axis = 1).reset_index(drop=True)
    ValidationDf = pd.concat([negativeValid, positiveValid]).drop(['companyid','marketmap', 'segment', 'subsegment'], axis = 1).reset_index(drop=True)
    TestDf = pd.concat([negativeTest, positiveTest]).drop(['companyid','marketmap', 'segment', 'subsegment'], axis = 1).reset_index(drop=True)

    # Outside prediction set
    df_predicting = df_predicting[df_predicting['patent_abstract'].str.len() > 0]
    ds_out = df_predicting[['patent_id', 'patent_abstract']].reset_index(drop=True)
    ds_out['label'] = 0
    ds_out['patent_abstract'] = ds_out['patent_abstract'].str.lower()


    ## Dataset generated
    datasets_ds = DatasetDict({
        "train": Dataset.from_pandas(TrainDf).class_encode_column('label').shuffle(),
        "test": Dataset.from_pandas(TestDf).class_encode_column('label').shuffle(),
        "valid": Dataset.from_pandas(ValidationDf).class_encode_column('label').shuffle(),
        "out": Dataset.from_pandas(ds_out).class_encode_column('label').shuffle(),
    })

    # print the size of each dataset
    print("Positive size: ", len(ds_positives))
    print("Negative size: ", len(ds_negatives))
    print("Train size: ", len(datasets_ds["train"]))
    print("Valid size: ", len(datasets_ds["valid"]))
    print("Test size: ", len(datasets_ds["test"]))
    print("Out size: ", len(datasets_ds["out"]))

    # Prepare the datasets
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    def tokenize_function(examples):
        return tokenizer(examples['patent_abstract'], padding="max_length", truncation=True, max_length=512)
    
    tokenized_datasets_ds = datasets_ds.map(tokenize_function, batched=True)

    # Model and training settings
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    def subsegment_name_processing(nameStr):
        nameStr = re.sub(r'[^\w\s]|_', '', nameStr)
        return nameStr.replace(" ", "")
    

    training_args = TrainingArguments(
        output_dir=output_dir/"model/SUBSEGMENT-{}".format(subsegment_name_processing(SELECTED_SUBSEGMENT)), 
        evaluation_strategy="epoch", 
        num_train_epochs=5,
        gradient_accumulation_steps=2, 
        per_device_train_batch_size=16,
        save_strategy='no', 
        logging_dir=output_dir/"model/SUBSEGMENT-{}/logs".format(subsegment_name_processing(SELECTED_SUBSEGMENT)),  # Directory to save TensorBoard logs
        logging_steps=100
    )

    ## Evaluation Metrics
    metric = evaluate.combine(["accuracy", "recall", "precision", "f1"])
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)
    ## Train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets_ds["train"],
        eval_dataset=tokenized_datasets_ds["valid"],
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.evaluate()

    # Save the evaluation metrics for the validation set
    trainer.save_metrics('eval', trainer.evaluate())

    # Evaluate the model on the test set
    test_eval_results = trainer.evaluate(eval_dataset=tokenized_datasets_ds["test"])

    # Save the evaluation metrics for the test set
    trainer.save_metrics('test', test_eval_results)

    ##trainer_downsampled.save_model("models_leftout/SUBSEGMENT-{}".format(subsegment_name_processing(SELECTED_SUBSEGMENT)))

    #Predict
    out_predictions = trainer.predict(tokenized_datasets_ds["out"])
    y_pred = out_predictions.predictions
    y_neg = out_predictions.predictions[:, 0]
    y_pos = out_predictions.predictions[:, 1]
    y_pred_dummy = np.argmax(y_pred, axis = -1)

    df=pd.DataFrame(data={'pred_pos':y_pos,'pred_neg':y_neg,'positive':y_pred_dummy,'patent_abstract':tokenized_datasets_ds["out"]["patent_abstract"],'patent_id':tokenized_datasets_ds["out"]["patent_id"]})

    df.to_csv(output_dir/"positive/{}_patent_positive_bert.csv".format(subsegment_name_processing(SELECTED_SUBSEGMENT)))
    ds_positives.to_csv(output_dir/"positive/{}_patent_training_TP.csv".format(subsegment_name_processing(SELECTED_SUBSEGMENT)))

    # predict on the test set
    test_predictions = trainer.predict(tokenized_datasets_ds["test"])
    y_pred = test_predictions.predictions
    y_neg = test_predictions.predictions[:, 0]
    y_pos = test_predictions.predictions[:, 1]
    y_pred_dummy = np.argmax(y_pred, axis = -1)

    df_test=pd.DataFrame(data={'pred_pos':y_pos,'pred_neg':y_neg,'positive':y_pred_dummy,'patent_abstract':tokenized_datasets_ds["test"]["patent_abstract"],'patent_id':tokenized_datasets_ds["test"]["patent_id"], 'true_label':tokenized_datasets_ds["test"]["label"]})
    

    df_test.to_csv(output_dir/"positive/{}_patent_test_bert.csv".format(subsegment_name_processing(SELECTED_SUBSEGMENT)))

    print(SELECTED_SUBSEGMENT+'finished')


In [7]:
# set working directory
os.chdir('PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_Patent/data/')
# read in patent data
g_patent = pd.read_csv('g_patent.tsv', sep='\t',low_memory=False) 
g_patent = g_patent[g_patent['patent_type'] == 'utility']

g_assignee = pd.read_csv('g_assignee_disambiguated.tsv', sep='\t',low_memory=False)
g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
g_assignee = g_assignee[['patent_id', 'assignee_id','disambig_assignee_organization']]
g_patent = g_patent.merge(g_assignee, on='patent_id', how='inner')
del g_assignee

pb_assignee = pd.read_excel('matched_assignee_companyname_medium.xlsx')

pb_marketmap = pd.read_csv('predicted_positive_v2.csv')
pb_marketmap.sort_values(by=['companyid'], inplace=True)

# group by companyid, count the unique values of marketmap
pb_marketmap['count_sector'] = pb_marketmap.groupby('companyid')['companyid'].transform('count')
# group by companyid, count the unique values of marketmap+segment combo
pb_marketmap['unique_segment'] = pb_marketmap['marketmap'] + pb_marketmap['segment']
pb_marketmap['count_segment'] = pb_marketmap.groupby('companyid')['unique_segment'].transform('nunique')
# groupy by companyid, count the unique values of marketmap
pb_marketmap['count_marketmap'] = pb_marketmap.groupby('companyid')['marketmap'].transform('nunique')

############## ADJACENT CONDITION ##############
pb_marketmap = pb_marketmap[pb_marketmap['count_marketmap'] == 1]
################################################

pb_marketmap = pd.merge(pb_marketmap, pb_assignee[['companyid','assignee_id','companyname']], on='companyid', how='inner')

df_training = pd.merge(g_patent, pb_marketmap, on='assignee_id', how='inner')
df_predicting = g_patent[~g_patent['patent_id'].isin(df_training['patent_id'])]
# get 50,000 from the predicting set
df_predicting = df_predicting.sample(50000, random_state=1)
# get the subsegments
ls_subsegment = df_training['fullname'].unique()
# sort the subsegment by name
ls_subsegment.sort()

In [14]:
# count the number of patents by fullname
patent_counts = df_training.groupby('fullname').size()
patent_counts

fullname
AI ML|AI & ML Semiconductors|Edge AI Software                   367
AI ML|AI & ML Semiconductors|Intelligent Sensors & Devices    14181
AI ML|AI & ML Semiconductors|Processor Design                 99799
AI ML|Autonomous Machines|Autonomous Vehicles                     1
AI ML|Autonomous Machines|Intelligent Robotics                 1621
                                                              ...  
Retail HealthTech|Virtual Health|Digital Therapeutics           194
Retail HealthTech|Virtual Health|telemedicine                  2623
Supply Chain Tech|Freight tech|Electric trucks                    3
Supply Chain Tech|Supply Chain Tech|Supply Chain Tech          5332
Supply Chain Tech|Warehousing tech|Sustainable packaging        191
Length: 239, dtype: int64

In [4]:
error_subseg_list = []
finished_subseg_df = pd.DataFrame()
for i in range(len(ls_subsegment)):   
    try:
        trainit(i, df_training, df_predicting, ls_subsegment)
        # generate a new observation and save to the finished_subseg_df
        finished_subseg_df = finished_subseg_df.append({'subsegment': ls_subsegment[i], 'finished': 1}, ignore_index=True)
    except:
        print('error in', i)
        error_subseg_list.append(i)

/var/folders/p9/3r6trd5n5t7127t0ljj8z63h0000gn/T/ipykernel_57954/3814075875.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_training.loc[:, 'label'] = 0


working on subsegment: AI ML|AI & ML Semiconductors|Edge AI Software


/var/folders/p9/3r6trd5n5t7127t0ljj8z63h0000gn/T/ipykernel_57954/3814075875.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ds_negatives_downsample = df_training[df_training.label == 0].groupby('segment', group_keys=False).apply(lambda x: x.sample(num_sampled, replace=True))


Stringifying the column:   0%|          | 0/3411 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/3411 [00:00<?, ? examples/s]

...

PATH_TO_LOCAL FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/1509 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

Map:   0%|          | 0/49993 [00:00<?, ? examples/s]